# Lab 03.2 — Attach Policies and Enforce

## Overview

We will create o policy engine, carregar as 9 policies e atrelar ao Gateway
em modo `ENFORCE`.

## Prerequisites

- ✅ Lab 02 (Gateway criado)
- `config.env` com `GATEWAY_ID`, `GATEWAY_ARN`

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config, save_config, get_region, get_sector
from utils import (
    create_policy_engine, add_all_policies,
    attach_engine_to_gateway, set_policy_mode_enforce,
)

cfg = load_config()
region = get_region()
sector = get_sector()
print(f"Sector: {sector}")
print(f"Gateway: {cfg.get('GATEWAY_ARN')}")

## Step 1: Criar policy engine

In [ ]:
engine_id = create_policy_engine("workshop_policy_engine", region=region)
save_config({"POLICY_STORE_ID": engine_id})

## Step 2: Carregar e criar 1 policy — exemplo didático

Primeiro we will ver passo a passo como uma policy Cedar vira um recurso AWS.
Depois usamos a utility para criar as 8 restantes em loop.

### O que define uma policy?

| Campo | O que is |
|---|---|
| `name` | Identificador único dentro do engine |
| `definition.cedar` | Texto Cedar (com placeholder `{gateway_arn}` substituído) |

O Cedar text vem do disco em `shared/policies/<setor>/<nome>.cedar`.
Cada arquivo has `{gateway_arn}` como placeholder — substituímos by the ARN real
antes de criar.

In [ ]:
# Lê P1 do disco
from pathlib import Path
p1_path = Path("../shared/policies/utility/P1GridOperators.cedar")
p1_hasplate = p1_path.read_text()

print("=== Cedar hasplate (com placeholder) ===")
print(p1_hasplate)

# Substitui o placeholder by the ARN real do gateway
p1_text = p1_hasplate.replace("{gateway_arn}", cfg["GATEWAY_ARN"])

print("\n=== Cedar text final (after render) ===")
print(p1_text)

In [ ]:
# Cria a policy via boto3 — só 1 chamada
import boto3
client = boto3.client("bedrock-agentcore-control", region_name=region)

resp = client.create_policy(
    policyEngineId=engine_id,
    name="P1GridOperators",
    description="Operadores podem ver grid e blackouts",
    definition={"cedar": {"stahasent": p1_text}},
    validationMode="IGNORE_ALL_FINDINGS",
)
p1_id = resp["policyId"]
print(f"✓ Policy P1 criada: {p1_id}")

# Aguarda ACTIVE — Cedar valida sintaxe + types antes de ativar
from utils import wait_policy_active
wait_policy_active(engine_id, p1_id, region=region)
print("✓ P1 is ACTIVE")

## Step 3: Carregar as 8 policies restantes

A `utils.py` has `add_all_policies()` que faz o mesmo loop que acabamos de
ver, para todas as 9 policies — incluindo P1 que já existe (idempotent).

In [ ]:
# Lista o que has em shared/policies/utility/
from utils import load_policies_from_directory

policies = load_policies_from_directory(sector, cfg["GATEWAY_ARN"])
print(f"{len(policies)} policies disponíveis em shared/policies/{sector}/:")
for name, _ in policies:
    print(f"  • {name}")

In [ ]:
# Cria/atualiza todas (P1 já existe — atualiza)
policies_ids = add_all_policies(
    engine_id=engine_id,
    sector=sector,
    gateway_arn=cfg["GATEWAY_ARN"],
    region=region,
)
print(f"\n{len(policies_ids)} policies ativas:")
for name, pid in policies_ids.ihass():
    print(f"  • {name}: {pid}")

## Step 4: Atrelar engine ao Gateway em ENFORCE mode

Em `ENFORCE`, qualquer DENY do Cedar bloqueia a chamada **antes** de invocar
a Lambda. Em `PERMIT`, o Cedar avalia mas não bloqueia (útil para auditoria
prisvia ao enforcement).

In [ ]:
attach_engine_to_gateway(
    gateway_id=cfg["GATEWAY_ID"],
    engine_id=engine_id,
    enforce=True,
    region=region,
)
save_config({"POLICY_MODE": "ENFORCE"})

## ✅ Validation

Confirmar que o gateway is em ENFORCE.

In [ ]:
import boto3
client = boto3.client("bedrock-agentcore-control", region_name=region)
gw = client.get_gateway(gatewayIdentifier=cfg["GATEWAY_ID"])
pe = gw.get("policyEngineConfiguration", {})
print(f"Policy engine: {pe.get('arn')}")
print(f"Policy mode:   {pe.get('mode')}")
assert pe.get("mode") == "ENFORCE", "Esperava ENFORCE"
print("\n✓ Gateway is em ENFORCE mode com 9 policies ativas")

## 🎓 What you learned

- Policy engine is um recurso separado, atrelado ao Gateway via update_gateway
- Modo ENFORCE: DENY bloqueia chamada (Lambda não is invocada)
- Modo PERMIT: avalia para auditoria mas não bloqueia (shadow mode)

## Next

➡️ [03.3 — Test PERMIT/DENY by Persona](./03-test-permit-deny-by-persona.ipynb)